# Industry classification
Predict industries from supplied text embeddings with logistic regression and k-NN, then combine their probabilities and calibrate confidence using financial evidence.

Run all cells from this folder with the two course datasets in `data/`. No GPU or embedding retraining is needed. Model settings are retained from the existing project; this notebook has no saved real-data results yet.

In [ ]:
import ast
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from threadpoolctl import threadpool_limits

threadpool_limits(limits=2)
started = perf_counter()


## 1. Data and independent splits
Join finances by company ID: medians resolve repeated financial rows; infinities become missing and valid negatives remain. Missing finances never remove companies.

Keep company IDs separate: approximately 81% fitting, 4.5% tuning, 4.5% confidence calibration and 10% final validation. Only tuning chooses the ensemble weight.

In [ ]:
financial_columns = ['net_profit_margin', 'ebitda_margin', 'asset_turnover']

def prepare_companies(companies, financials=None):
    if financials is not None:
        financials = financials.set_index('id').reindex(columns=financial_columns).replace([np.inf, -np.inf], np.nan)
        financials = financials.groupby('id').median()
        companies = companies.drop(columns=financial_columns, errors='ignore').merge(
            financials, on='id', how='left', validate='many_to_one', sort=False)
    companies = companies.reindex(columns=companies.columns.union(financial_columns, sort=False)).copy()
    companies[financial_columns] = companies[financial_columns].replace([np.inf, -np.inf], np.nan)
    return companies

df = prepare_companies(pd.read_pickle('data/df_train.pkl'), pd.read_pickle('data/df_financials_train.pkl'))

def split(companies, size):
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=size, random_state=42).split(
        companies, groups=companies['id']))
    return companies.iloc[a], companies.iloc[b]

df_train, df_val = split(df, 0.1)
df_fit, df_development = split(df_train, 0.1)
df_tune, df_cal = split(df_development, 0.5)
display(pd.Series([len(d) for d in [df_fit, df_tune, df_cal, df_val]],
                  index=['Fitting', 'Tuning', 'Calibration', 'Validation'], name='Records'))

def embeddings(companies):
    values = companies['business_description_embedding'].map(
        lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
    return np.array(values.tolist(), dtype=np.float32)

X_fit, X_tune, X_cal, X_val = [embeddings(d) for d in [df_fit, df_tune, df_cal, df_val]]
le = LabelEncoder().fit(df_fit['industry'])
assert set(df_tune['industry']) | set(df_cal['industry']) | set(df_val['industry']) <= set(le.classes_)
y_fit, y_tune, y_cal, y_val = [le.transform(d['industry']) for d in [df_fit, df_tune, df_cal, df_val]]
print('Embedding dimensions:', X_fit.shape[1], '| Industries:', len(le.classes_))


## 2. Class balance and PCA
PCA learns 128 components from fitting embeddings only. k-NN uses all 128; the plot shows just the first two. Overlap in this compressed view does not rule out signal in the original space.

In [ ]:
counts = df_train['industry'].value_counts()
counts.sort_values().plot.barh(figsize=(9, 6), title='Industry distribution in training data')
plt.xlabel('Number of companies')
plt.ylabel('Industry')
plt.tight_layout()
plt.show()

pca = PCA(n_components=128, random_state=42)
X_fit_pca = pca.fit_transform(X_fit)
X_tune_pca, X_cal_pca, X_val_pca = [pca.transform(X) for X in [X_tune, X_cal, X_val]]
plt.figure(figsize=(10, 6))
colors = sns.color_palette('husl', len(le.classes_))
for label, industry in enumerate(le.classes_):
    mask = y_fit == label
    plt.scatter(X_fit_pca[mask, 0], X_fit_pca[mask, 1], s=6, alpha=0.3, color=colors[label], label=industry)
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('Fitting embeddings: first two PCA components')
plt.legend(bbox_to_anchor=(1.02, 1), fontsize=7, markerscale=3)
plt.tight_layout()
plt.show()
print('Variance retained (2 / 128 components):', pca.explained_variance_ratio_[:2].sum(),
      pca.explained_variance_ratio_.sum())


## 3. Classifiers and soft voting
Logistic regression uses the original embeddings. Distance-weighted k-NN uses PCA coordinates. Choose the logistic-regression voting weight on tuning F1, before seeing calibration or validation outcomes.

In [ ]:
lr = LogisticRegression(C=10, max_iter=1000).fit(X_fit, y_fit)
knn = KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=2).fit(X_fit_pca, y_fit)
assert np.array_equal(lr.classes_, knn.classes_) and np.array_equal(lr.classes_, np.arange(len(le.classes_)))
proba_tune_lr, proba_cal_lr, proba_val_lr = [lr.predict_proba(X) for X in [X_tune, X_cal, X_val]]
proba_tune_knn, proba_cal_knn, proba_val_knn = [knn.predict_proba(X) for X in [X_tune_pca, X_cal_pca, X_val_pca]]
weights = [0.25, 0.5, 0.75]
tuning_f1 = [f1_score(y_tune, (w * proba_tune_lr + (1 - w) * proba_tune_knn).argmax(axis=1),
                      average='weighted') for w in weights]
lr_weight = weights[np.argmax(tuning_f1)]
display(pd.DataFrame({'LR weight': weights, 'Tuning weighted F1': tuning_f1}))
print('Selected LR weight:', lr_weight)


## 4. Financial evidence and confidence
Use fitting-only financial percentiles: values near the predicted industry's middle are more compatible than tail values. This bounded score tolerates outliers without assuming normality; it is not a probability of correct classification or a formal p-value.

A separate binary logistic model learns correctness on calibration companies. Its five inputs are maximum vote, gap to second place, model agreement, financial compatibility and financial coverage. A text-only calibrator provides a comparison.

In [ ]:
profiles = {(industry, column): np.sort(df_fit.loc[df_fit['industry'] == industry, column].dropna())
            for industry in le.classes_ for column in financial_columns}

def confidence_features(companies, proba_lr, proba_knn):
    votes = lr_weight * proba_lr + (1 - lr_weight) * proba_knn
    prediction = votes.argmax(axis=1)
    top = np.sort(votes, axis=1)
    predicted_industries = le.inverse_transform(prediction)
    total, available = np.zeros(len(companies)), np.zeros(len(companies))
    for industry in le.classes_:
        for column in financial_columns:
            reference = profiles[industry, column]
            if len(reference) < 20:
                continue
            values = companies[column].to_numpy()
            mask = (predicted_industries == industry) & np.isfinite(values)
            left = np.searchsorted(reference, values[mask], side='left')
            right = np.searchsorted(reference, values[mask], side='right')
            percentile = (0.5 * (left + right) + 0.5) / (len(reference) + 1)
            total[mask] += 2 * np.minimum(percentile, 1 - percentile)
            available[mask] += 1
    compatibility = np.divide(total, available, out=np.full(len(companies), 0.5), where=available > 0)
    features = np.column_stack([top[:, -1], top[:, -1] - top[:, -2],
                               proba_lr.argmax(axis=1) == proba_knn.argmax(axis=1),
                               compatibility, available / len(financial_columns)])
    return prediction, features

pred_cal, features_cal = confidence_features(df_cal, proba_cal_lr, proba_cal_knn)
correct_cal = (pred_cal == y_cal).astype(int)
text_calibrator = LogisticRegression().fit(features_cal[:, :3], correct_cal)
finance_calibrator = LogisticRegression().fit(features_cal, correct_cal)

def predict_industry(companies, financials=None):
    companies = prepare_companies(companies, financials)
    X = embeddings(companies)
    prediction, features = confidence_features(companies, lr.predict_proba(X), knn.predict_proba(pca.transform(X)))
    return le.inverse_transform(prediction), finance_calibrator.predict_proba(features)[:, 1]


## 5. Final validation: required models
Report accuracy and weighted/macro F1. Per-industry recall describes errors; normalized confusion matrices show which industries are confused. Compare shared errors to assess whether voting has complementary information. All model choices are already fixed.

In [ ]:
predictions = {'Logistic regression': proba_val_lr.argmax(axis=1), 'k-NN': proba_val_knn.argmax(axis=1)}
results = []
for name, prediction in predictions.items():
    results.append([name, accuracy_score(y_val, prediction), f1_score(y_val, prediction, average='weighted'),
                    f1_score(y_val, prediction, average='macro')])
    report = pd.DataFrame(classification_report(y_val, prediction, labels=np.arange(len(le.classes_)),
                         target_names=le.classes_, output_dict=True, zero_division=0)).T.loc[le.classes_]
    print(name)
    display(report.sort_values('f1-score'))
    print('Industry size / error-rate correlation:', counts.reindex(report.index).corr(1 - report['recall']))
    cm = confusion_matrix(y_val, prediction, labels=np.arange(len(le.classes_)))
    pairs = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    np.fill_diagonal(pairs.values, 0)
    display(pairs.stack().sort_values(ascending=False).head(5).rename('Misclassifications'))
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix(y_val, prediction, labels=np.arange(len(le.classes_)), normalize='true'),
                xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues', vmin=0, vmax=1)
    plt.xlabel('Predicted industry')
    plt.ylabel('True industry')
    plt.title(name + ': fraction of each true industry')
    plt.tight_layout()
    plt.show()
comparison = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Weighted F1', 'Macro F1']).set_index('Model')
display(comparison)
print('Higher validation weighted F1:', comparison['Weighted F1'].idxmax())
errors_lr, errors_knn = predictions['Logistic regression'] != y_val, predictions['k-NN'] != y_val
print('Both wrong / only LR wrong / only k-NN wrong:',
      (errors_lr & errors_knn).sum(), (errors_lr & ~errors_knn).sum(), (~errors_lr & errors_knn).sum())


## 6. Official score and financial distributions
The assignment score is **weighted F1 × (1 − Brier)**, with Brier = mean((confidence − correctness)²). Confidence does not weight the F1 samples. All ensemble variants predict the same labels; finance changes confidence only. The financial variant is the fixed final candidate, not selected using this validation table.

Boxplots use fitting data; hiding outlier dots does not delete observations. Symmetric log axes retain negative values.

In [ ]:
def confidence_score(y_true, y_pred, confidence):
    y_true, y_pred, confidence = np.asarray(y_true), np.asarray(y_pred), np.asarray(confidence, dtype=float)
    assert len(y_true) == len(y_pred) == len(confidence)
    assert np.all((confidence >= 0) & (confidence <= 1))
    return f1_score(y_true, y_pred, average='weighted') * (1 - np.mean((confidence - (y_pred == y_true)) ** 2))

pred_val, features_val = confidence_features(df_val, proba_val_lr, proba_val_knn)
confidence_methods = {'Raw ensemble': features_val[:, 0],
                      'Text-calibrated ensemble': text_calibrator.predict_proba(features_val[:, :3])[:, 1],
                      'Financial-calibrated ensemble': finance_calibrator.predict_proba(features_val)[:, 1]}
methods = {name: (pred_val, c) for name, c in confidence_methods.items()}
methods['Logistic regression'] = (predictions['Logistic regression'], proba_val_lr.max(axis=1))
methods['k-NN'] = (predictions['k-NN'], proba_val_knn.max(axis=1))
bonus_results = []
for name, (prediction, confidence) in methods.items():
    bonus_results.append([name, f1_score(y_val, prediction, average='weighted'),
                          np.mean((confidence - (prediction == y_val)) ** 2),
                          confidence_score(y_val, prediction, confidence)])
display(pd.DataFrame(bonus_results, columns=['Method', 'Weighted F1', 'Brier', 'Official score']).set_index('Method'))
display(df_fit.groupby('industry')[financial_columns].agg(['count', 'median']))
for column in financial_columns:
    plt.figure(figsize=(9, 6))
    sns.boxplot(data=df_fit, x=column, y='industry', showfliers=False)
    plt.xscale('symlog', linthresh=0.1)
    plt.xlabel(column.replace('_', ' ') + ' (symmetric log scale)')
    plt.ylabel('Industry')
    plt.title('Fitting financial distributions: ' + column.replace('_', ' '))
    plt.tight_layout()
    plt.show()


## 7. Reliability, review and prediction
Compare mean confidence with actual accuracy in bins, including counts. Flag confidence below 0.8 as a starting review policy; flagged records still count in every metric.

`predict_industry(companies, financials=None)` returns `(industry_names, confidence)` in input-row order, without reading targets. Supply prejoined financial columns or a separate finance table; absent ratios become NaN. ID-disjoint validation can still share descriptions with fitting: report both groups separately.

In [ ]:
confidence_val = confidence_methods['Financial-calibrated ensemble']
outcomes = pd.DataFrame({'Confidence': confidence_val, 'Correct': pred_val == y_val})
outcomes['Bin'] = pd.cut(confidence_val, np.linspace(0, 1, 11), include_lowest=True)
reliability = outcomes.groupby('Bin', observed=True).agg(
    Count=('Correct', 'size'), Mean_confidence=('Confidence', 'mean'), Accuracy=('Correct', 'mean'))
display(reliability)
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
plt.plot(reliability['Mean_confidence'], reliability['Accuracy'], 'o-', label='Validation bins')
plt.xlabel('Mean predicted confidence')
plt.ylabel('Observed accuracy')
plt.title('Confidence and accuracy')
plt.legend()
plt.tight_layout()
plt.show()
review = confidence_val < 0.8
outcomes['Human review'] = review
display(outcomes.groupby('Human review').agg(Count=('Correct', 'size'), Accuracy=('Correct', 'mean')))
print('Flagged share:', review.mean(), '| Errors captured:', np.sum(review & (pred_val != y_val)),
      'of', np.sum(pred_val != y_val))
example = df_val.drop(columns='industry').head(3)
industries, confidences = predict_industry(example)
display(pd.DataFrame({'ID': example['id'].to_numpy(), 'Predicted industry': industries, 'Confidence': confidences}))
seen_embedding = df_val['business_description_embedding'].isin(df_fit['business_description_embedding'])
display(pd.DataFrame({'Embedding seen during fitting': seen_embedding.to_numpy(), 'Correct': pred_val == y_val})
        .groupby('Embedding seen during fitting').agg(Count=('Correct', 'size'), Accuracy=('Correct', 'mean')))
print('Whole notebook elapsed seconds:', round(perf_counter() - started, 1))


## Interpretation and limits
Use the tables to compare the required classifiers by weighted F1; inspect macro F1 and per-industry errors to avoid hiding weak minority classes. PCA shows only part of the available variance, and financial distributions overlap. Voting and calibration must earn their usefulness in the reported comparisons.

In the archived real-data run, Materials represented 26.5% of training records while the smallest industry had 160 records. The PCA scatter showed concentrations but substantial central overlap; two components retained 25.3% of variance. LR led k-NN on weighted F1 (77.78% vs 77.14%); lower macro F1 (71.75% / 71.05%) highlighted weaker small-class performance. Both often confused Capital Goods with Materials; larger classes tended to have fewer errors, but class size was not the only explanation. Voting reached 78.09% F1; financial calibration reached 67.50% official score, only 0.03 percentage points above text calibration. These are historical findings, not a new run.

Historical outputs from the previous version are preserved in `archive/`, not copied here as new measurements. Real-data scores, plot observations and laptop runtime must be checked after a fresh run. No private-test score or bonus is guaranteed. The official score cannot exceed weighted F1. AI assistance was used; the study notebook and guide explain the decisions.